In [1]:
import torch
from datasets import load_dataset
import librosa
import pandas as pd
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = 'mms_1b_mansi'

stt_model_path = f"../stt/models/{model_name}"

/home/dev/workspace/shidlovskiy/mansi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-26 19:12:13.327589: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
tts_path = "../data/tts_dataset"
tts_test_dataset = load_dataset('csv', data_files=tts_path+'/test/metadata.csv', delimiter='\t', split='train')

stt_path = "../data/stt_dataset"
stt_test_dataset = load_dataset('csv', data_files=stt_path+'/test/metadata.csv', delimiter='\t', split='train')

bonus_path = "../data/bonus_dataset"
bonus_test_dataset = load_dataset('csv', data_files=bonus_path+'/test/metadata.csv', delimiter='\t', split='train')

In [3]:
import evaluate
import re
import unicodedata

def normalize(text: str) -> str:
    text = text.lower()
    
    normalized_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat == "Zs":
            normalized_chars.append(ch)
        elif unicodedata.combining(ch) and ch in "̄":
            normalized_chars.append(ch)
        elif ch in ["ӈ"]:
            normalized_chars.append(ch)

    return re.sub(r"\s+", " ", "".join(normalized_chars)).strip()

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")


def compute_metrics(pred_text, test_text):
    orh_wer = 100 * metric_wer.compute(predictions=pred_text, references=test_text)
    orh_cer = 100 * metric_cer.compute(predictions=pred_text, references=test_text)

    pred_str_norm = [normalize(pred) for pred in pred_text]
    label_str_norm = [normalize(label) for label in test_text]

    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [ 
        label_str_norm[i] for i in range(len(label_str_norm))if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric_wer.compute(predictions=pred_str_norm, references=label_str_norm)
    cer = 100 * metric_cer.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer": wer, "cer": cer, "orh_wer": orh_wer, "orh_cer": orh_cer}

In [4]:
wer_list = []
cer_list = []
orh_wer_list = []
orh_cer_list = []



processor = Wav2Vec2Processor.from_pretrained(stt_model_path)
model = Wav2Vec2ForCTC.from_pretrained(
    stt_model_path,
    ctc_loss_reduction="mean", 
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size = processor.tokenizer.vocab_size
).to(device)
model.eval()

dataset = pd.DataFrame(columns=['model_out', 'label', 'wer', 'cer', 'orh_wer', 'orh_cer'])

for sample in tts_test_dataset:
    pred_file, sr = librosa.load(f'{tts_path}/test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    metrics = compute_metrics([pred_text], [test_text])

    wer_list.append(metrics['wer'])
    cer_list.append(metrics['cer'])
    orh_wer_list.append(metrics['orh_wer'])
    orh_cer_list.append(metrics['orh_cer'])
    new_row = pd.DataFrame([{
        'model_out': pred_text, 
        'label': test_text, 
        'wer': metrics['wer'], 
        'cer': metrics['cer'], 
        'orh_wer': metrics['orh_wer'], 
        'orh_cer': metrics['orh_cer']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)


for sample in stt_test_dataset:
    pred_file, sr = librosa.load(f'{stt_path}/test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    metrics = compute_metrics([pred_text], [test_text])
    new_row = pd.DataFrame([{
        'model_out': pred_text, 
        'label': test_text, 
        'wer': metrics['wer'], 
        'cer': metrics['cer'], 
        'orh_wer': metrics['orh_wer'], 
        'orh_cer': metrics['orh_cer']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)


for sample in bonus_test_dataset:
    pred_file, sr = librosa.load(f'{bonus_path}/test/wavs/{sample['audio_file']}', sr=16000)
    test_text = sample['text']

    inputs = processor(pred_file, sampling_rate=16000, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]

    metrics = compute_metrics([pred_text], [test_text])
    new_row = pd.DataFrame([{
        'model_out': pred_text, 
        'label': test_text, 
        'wer': metrics['wer'], 
        'cer': metrics['cer'], 
        'orh_wer': metrics['orh_wer'], 
        'orh_cer': metrics['orh_cer']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)

dataset.to_csv(f'predicts_{model_name}.csv', index=False, sep="\t")

/tmp/ipykernel_47986/2059976553.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dataset = pd.concat([dataset, new_row], ignore_index=True)
